# 02b – Design experiment: oversampling rare labels
The first LoRA run (`lora`, notebook 02) never predicted the rare labels `excluded`, `not included` and `not applicable`
(8, 13 and 28 of 602 training rows), so its val macro-F1 was only 0.33.
This run repeats rare-label training rows (factor = round(60/count), capped at 6) and keeps everything else the same.
The model with the better **val macro-F1** is used for the final test. Test data is not touched.

In [ ]:
# ==== EDIT THESE TWO LINES ====
GITHUB_USER = "FatimaRana50"
GIT_NAME    = "Fatima Rana"
# ==============================
REPO = "trialmatch"

!pip -q install peft datasets
!pip -q uninstall -y torchao       # Colab ships torchao 0.10; peft refuses to load with it (we do not use it)
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
token = userdata.get('GH_TOKEN')          # Colab sidebar -> key icon -> add secret GH_TOKEN
if not os.path.exists(f'/content/{REPO}'):
    !git clone -q https://{token}@github.com/{GITHUB_USER}/{REPO}.git /content/{REPO}
%cd /content/{REPO}
!git pull -q
!git config user.name "{GIT_NAME}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
import sys; sys.path.insert(0, '/content/trialmatch/src')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
CKPT_DIR = "/content/drive/MyDrive/trialmatch/lora_r16_os"
os.makedirs(CKPT_DIR, exist_ok=True)

In [ ]:
!python src/train.py --out {CKPT_DIR} --oversample 60 --tag os

In [ ]:
from IPython.display import Image, display
display(Image("results/training_curves_os.png"))
print(open("logs/train_config_os.json").read())

In [ ]:
!python src/evaluate.py --split val --tag lora_os --adapter {CKPT_DIR}/final_adapter

In [ ]:
import json, pandas as pd
rows = []
for tag in ["base", "lora", "lora_os"]:
    m = json.load(open(f"results/{tag}_val_metrics.json"))
    rows.append({"model": tag,
                 "acc": m["overall"]["classification"]["accuracy"],
                 "macroF1": m["overall"]["classification"]["macro_f1"],
                 "incl_acc": m["inclusion"]["classification"]["accuracy"],
                 "excl_acc": m["exclusion"]["classification"]["accuracy"],
                 "evid_micro_F1": m["overall"]["evidence"]["micro_f1"],
                 "evid_macro_F1": m["overall"]["evidence"]["macro_f1"],
                 **{f"F1[{l}]": c["f1"] for l, c in m["overall"]["classification"]["per_class"].items()}})
cmp = pd.DataFrame(rows).round(3); print(cmp.T.to_string())
cmp.to_csv("results/design_comparison_val.csv", index=False)

In [ ]:
!pip freeze > logs/colab_env_02b.txt
!git add results/ logs/ && git commit -m "Oversampling experiment on val" && git pull -q --rebase && git push